In [ ]:
import pandas as pd
import numpy as np
import time
import pickle 
import torch

from sbi.utils import BoxUniform
from scipy.integrate import odeint
from sbi.utils import MultipleIndependent

import matplotlib.pyplot as plt

_ = np.random.seed(0)
_ = torch.manual_seed(0)

In [ ]:
with open('../../Data/Model1/M1_dataset.pkl', 'rb') as handle:
    sim_traj = pickle.load(handle)

with open('../../Data/Model1/M1_params.pkl', 'rb') as handle:
    sim_params = pickle.load(handle)
    
with open('../../Data/Model1/sim_dataset.pkl', 'rb') as f:
    simulation_dataset = pickle.load(f)

In [ ]:
def poisson_noise(simulation):
    
    with_noise = np.random.poisson(np.maximum(simulation, 1e-6))

    return with_noise

In [ ]:
traj   = torch.tensor(poisson_noise(sim_traj.numpy()), dtype=torch.float32)
params = sim_params

In [ ]:
param_names=['beta','kappa','gamma']

In [ ]:
from scipy.integrate import odeint
import torch

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

In [ ]:
import torch.nn as nn
from sbi.inference import NPE
from sbi.neural_nets import posterior_nn

class LSTMembedding(nn.Module):
    def __init__(self, input_dim=1, hidden_dim=128, output_dim=30, num_layers=1,bidirectional=True):
        super().__init__()

        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, bidirectional=bidirectional, batch_first = True)
        self.fc = nn.Linear(hidden_dim * (2 if bidirectional else 1), output_dim)
        self.tanh = nn.Tanh()
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(-1)
        lstm_out, _ = self.lstm(x)  
        last_hidden = lstm_out[:, -1, :]
        last_hidden = self.dropout(last_hidden)
        out = self.fc(last_hidden)   
        return out

In [ ]:
embedding_net =LSTMembedding(input_dim=1, hidden_dim=128, output_dim=30).to(device)
embedding_net

In [ ]:
low  = torch.tensor([0.01, 0.01, 0.01])
high = torch.tensor([1.5, 0.5, 0.5])

prior = BoxUniform(low=low, high=high)

In [ ]:
xs     = traj
thetas = params

In [ ]:
neural_posterior = posterior_nn(model='maf', embedding_net=embedding_net)
inference = NPE(density_estimator=neural_posterior,device=device)

density_estimator=inference.append_simulations(thetas, xs).train(training_batch_size=256)

posterior= inference.build_posterior(density_estimator)

In [ ]:
posterior_samples=[]

for i, sim in enumerate(simulation_dataset):  
    x_obs = torch.as_tensor(sim['poisson'], dtype=torch.float32).to(device)
    
    samples = posterior.sample((10000,), x=x_obs)
    samples_df=pd.DataFrame(samples.cpu().numpy(),columns=param_names)
    posterior_samples.append(samples_df)